# 02 — Da seleção fotométrica à validação espectroscópica

Workshop SAB — AI-Scope / ADSS

**Pergunta científica:** depois de selecionar candidatos em um catálogo, quais deles possuem informação espectroscópica independente?

O exemplo usa uma tabela local de alvos e faz o crossmatch no servidor com DESI DR1. Isso evita baixar um catálogo espectroscópico inteiro apenas para cruzá-lo localmente.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import adss

BASE_URL = "https://ai-scope.cbpf.br/"
USERNAME = os.getenv("AI_SCOPE_USERNAME")
PASSWORD = os.getenv("AI_SCOPE_PASSWORD")

if not USERNAME or not PASSWORD:
    raise RuntimeError("Defina AI_SCOPE_USERNAME e AI_SCOPE_PASSWORD no ambiente.")

cl = adss.ADSSClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD)

In [ ]:
# Use a saída do notebook anterior ou substitua por sua própria lista de alvos.
# Exemplo mínimo para permitir execução independente:
targets = pd.DataFrame({
    "target_name": ["target_1", "target_2", "target_3"],
    "ra": [150.10, 150.20, 150.30],
    "dec": [2.10, 2.20, 2.30],
})
targets

In [ ]:
query = """
SELECT
    u.target_name,
    u.ra AS input_ra,
    u.dec AS input_dec,
    d.id AS desi_id,
    d.ra AS desi_ra,
    d.dec AS desi_dec,
    d.z,
    d.zerr,
    d.zwarn,
    d.spectype,
    d.subtype,
    d.deltachi2,
    dist(u.ra, u.dec, d.ra, d.dec) AS separation_deg
FROM upload.targets AS u
JOIN desi.desi_specz_dr1 AS d
  ON cone(d.ra, d.dec, u.ra, u.dec, 0.0003)
"""

spec = cl.query_and_wait(
    query_text=query,
    mode="astroql",
    file=targets,
    table_name="targets",
).data
spec.head()

In [ ]:
spec["separation_arcsec"] = spec.separation_deg * 3600
print(spec.spectype.value_counts(dropna=False))

fig, ax = plt.subplots(figsize=(6,4))
ax.hist(spec.separation_arcsec.dropna(), bins=30)
ax.set(xlabel="Crossmatch separation [arcsec]", ylabel="N")
plt.show()

### Investigação
Compare a classificação/seleção original dos alvos com `spectype`, `z`, `zwarn` e `deltachi2`. Em um projeto real, este passo pode transformar um conjunto de candidatos puramente fotométricos em uma amostra de validação independente.